<a href="https://colab.research.google.com/github/lightcode01-oss/IYB_INTERNSHIP/blob/main/Day_11_KNN_Classification_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DAY 11 — K-Nearest Neighbors Classification Model

## YBI Foundation — Python, AI & Data Science Internship

### Objective

The objective of Day 11 is to train and evaluate a K-Nearest Neighbors
(KNN) classification model for predicting student performance categories.

KNN is a distance-based algorithm. Therefore, this day will also introduce
feature scaling into the preprocessing pipeline.

### Today's Work

- Load the dataset independently.
- Recreate the project target.
- Prevent target leakage.
- Prepare numerical, ordinal and categorical features.
- Perform a stratified train-test split.
- Apply one-hot encoding to categorical features.
- Apply feature scaling to numerical features.
- Train a KNN classifier.
- Generate predictions.
- Evaluate the model.
- Analyze the confusion matrix.
- Compare KNN with previous baseline models.
- Save the model comparison results.

No hyperparameter tuning will be performed today.

In [1]:
!pip -q install ucimlrepo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
student_performance = fetch_ucirepo(id=320)

X = student_performance.data.features
y_original = student_performance.data.targets

df = X.copy()
df["G3"] = y_original["G3"]

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

Dataset loaded successfully.
Dataset shape: (649, 31)


In [3]:
def performance_category(grade):
    if grade < 10:
        return "Low"
    elif grade < 15:
        return "Medium"
    else:
        return "High"

df["performance_category"] = df["G3"].apply(
    performance_category
)

print(df["performance_category"].value_counts())

performance_category
Medium    418
High      131
Low       100
Name: count, dtype: int64


In [4]:
target = "performance_category"

X = df.drop(columns=[target])
y = df[target]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

Feature shape: (649, 31)
Target shape: (649,)

Target distribution:
performance_category
Medium    418
High      131
Low       100
Name: count, dtype: int64


In [5]:
X_model = X.drop(columns=["G3"])

print("G3 present in model features:", "G3" in X_model.columns)
print("Final model feature count:", X_model.shape[1])

G3 present in model features: False
Final model feature count: 30


In [6]:
numerical_features = [
    "age",
    "absences"
]

ordinal_features = [
    "Medu",
    "Fedu",
    "traveltime",
    "studytime",
    "failures",
    "famrel",
    "freetime",
    "goout",
    "Dalc",
    "Walc",
    "health"
]

binary_features = [
    "school",
    "sex",
    "address",
    "famsize",
    "Pstatus",
    "schoolsup",
    "famsup",
    "paid",
    "activities",
    "nursery",
    "higher",
    "internet",
    "romantic"
]

nominal_features = [
    "Mjob",
    "Fjob",
    "reason",
    "guardian"
]

categorical_features = (
    binary_features +
    nominal_features
)

print("Numerical features:", len(numerical_features))
print("Ordinal features:", len(ordinal_features))
print("Categorical features:", len(categorical_features))

Numerical features: 2
Ordinal features: 11
Categorical features: 17


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 519
Testing samples: 130


## Why Scaling Is Required for KNN

KNN calculates distances between observations.

Features with larger numerical ranges can have a stronger influence on
distance calculations.

For example, `absences` can have a much larger numerical range than some
rating-style features.

Therefore, numerical and ordinal features will be standardized before KNN
uses them.

The scaling operation will be placed inside the Scikit-learn Pipeline so that
the scaler is fitted only on the training data.

In [8]:
categorical_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

numerical_scaler = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_encoder,
            categorical_features
        ),
        (
            "numerical",
            numerical_scaler,
            numerical_features + ordinal_features
        )
    ],
    remainder="drop"
)

print("KNN preprocessing pipeline created.")

KNN preprocessing pipeline created.


In [9]:
knn_model = KNeighborsClassifier(
    n_neighbors=5
)

print("KNN classifier created successfully.")
print("Number of neighbors:", knn_model.n_neighbors)

KNN classifier created successfully.
Number of neighbors: 5


In [10]:
knn_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", knn_model)
    ]
)

print("Complete KNN pipeline created successfully.")

Complete KNN pipeline created successfully.


In [11]:
knn_pipeline.fit(
    X_train,
    y_train
)

print("KNN model trained successfully.")

KNN model trained successfully.


In [12]:
y_pred = knn_pipeline.predict(X_test)

print("Predictions generated successfully.")

print("\nFirst 20 predictions:")
print(y_pred[:20])

Predictions generated successfully.

First 20 predictions:
['Medium' 'Medium' 'Medium' 'Medium' 'Low' 'Medium' 'Medium' 'Medium'
 'Medium' 'Low' 'High' 'High' 'Medium' 'Medium' 'Medium' 'Medium' 'Medium'
 'Medium' 'Medium' 'Medium']


In [13]:
prediction_comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

prediction_comparison.head(20)

,Actual,Predicted
0,Medium,Medium
1,Low,Medium
2,Medium,Medium
3,Low,Medium
4,Medium,Low
5,Medium,Medium
6,High,Medium
7,Medium,Medium
8,Medium,Medium
9,Low,Low


In [14]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

print("KNN Accuracy:", round(accuracy, 4))
print("KNN Accuracy (%):", round(accuracy * 100, 2))

KNN Accuracy: 0.6385
KNN Accuracy (%): 63.85


In [15]:
precision = precision_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

print("KNN Macro Precision:", round(precision, 4))

KNN Macro Precision: 0.5523


In [16]:
f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

print("KNN Macro F1:", round(f1, 4))

KNN Macro F1: 0.4931


In [18]:
recall = recall_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

knn_results = pd.DataFrame({
    "Model": ["KNN"],
    "Accuracy": [accuracy],
    "Macro Precision": [precision],
    "Macro Recall": [recall],
    "Macro F1": [f1]
})

knn_results

,Model,Accuracy,Macro Precision,Macro Recall,Macro F1
0,KNN,0.638462,0.552284,0.472527,0.493093


In [19]:
print(
    classification_report(
        y_test,
        y_pred,
        labels=["Low", "Medium", "High"],
        zero_division=0
    )
)

              precision    recall  f1-score   support

         Low       0.50      0.25      0.33        20
      Medium       0.68      0.82      0.75        84
        High       0.47      0.35      0.40        26

    accuracy                           0.64       130
   macro avg       0.55      0.47      0.49       130
weighted avg       0.61      0.64      0.61       130

